In [ ]:
# In practice dataset will store something like:

# duration_i = observed minutes

# status_i = 1 if unplug seen, 0 if censored

#EV under ToU vs. EV under 2-rates tariffs, potentially enriched with additional out of trial V2G data

# Initialisation

In [ ]:
import polars as pl
import numpy as np
import pandas as pd 

import lifelines as ll
import matplotlib.pyplot as plt

## Functions

### preprocessing functions

In [ ]:
def clean_df(data: str, pretrial: bool = True):
    
    path = f"../data/{data}PreConsolide.csv" if pretrial else f"../data/{data}ConsolideNEW.csv"

    df = pl.read_csv(path)

    time_col = df.columns[0]  # the unnamed time column (first column)

    df = (
        df.with_columns(
            [
                pl.col(time_col).str.strptime(pl.Datetime, strict=False).alias("ts"),
                pl.exclude(time_col).cast(pl.Float64),
            ]
        )
    ).drop([df.columns[0]])

    df = (
    df.unpivot(index=["ts"])
      .rename({"variable": "household", "value": "kwh"})
      .sort(["household", "ts"])
      .with_columns([
          pl.col("ts")
            .dt.strftime("%w")
            .cast(pl.Int8)
            .is_in([0, 6])
            .cast(pl.Int8)
            .alias("is_weekend"),

          pl.col("ts").dt.month().alias("month"),
      ])
      .with_columns([
          # season
          pl.when(pl.col("month").is_in([12, 1, 2]))
            .then(pl.lit("winter"))
            .when(pl.col("month").is_in([3, 4, 5]))
            .then(pl.lit("spring"))
            .when(pl.col("month").is_in([6, 7, 8]))
            .then(pl.lit("summer"))
            .otherwise(pl.lit("autumn"))
            .alias("season"),

          # time period
          pl.when(pl.col("is_weekend") == 1)
            # WEEKEND
            .then(
                pl.when(pl.col("ts").dt.hour() < 8)
                  .then(pl.lit("offpeak"))
                  .when(pl.col("ts").dt.hour().is_between(16, 20))
                  .then(pl.lit("peak"))
                  .otherwise(pl.lit("mid"))
            )
            # WEEKDAY
            .otherwise(
                pl.when(pl.col("ts").dt.hour() < 8)
                  .then(pl.lit("offpeak"))
                  .otherwise(pl.lit("mid"))
            )
            .alias("time_period"),
      ])
      .drop("month")
)
    
    df = df.with_columns(
    pl.col("kwh").fill_null(0)
)

    return df


In [ ]:
# Remove inactive peiods : vacation, metering issue...

def remove_inactive(df_interest, INACTIVE_KWH = 0.05, PROLONGED_DAYS = 10, MIN_MAX_KWH = 1.0):

    
# Keep only households whose max consumption reaches at least 1 kWh
    valid_households = (
        df_interest
        .group_by("household")
        .agg(pl.col("kwh").max().alias("max_kwh"))
        .filter(pl.col("max_kwh") >= MIN_MAX_KWH)
        .select("household")
    )

    df_interest = df_interest.join(valid_households, on="household", how="inner")

    df = df_interest.with_columns(
        (pl.col("kwh") < INACTIVE_KWH).alias("inactive")
    )

    inactive_runs = (
        df
        .sort(["household", "ts"])
        .with_columns(
            (
                pl.col("inactive")
                &
                ~pl.col("inactive").shift(1).over("household").fill_null(False) #“Was the previous record in this household active?”
            ).cast(pl.Int8).alias("start_run")
        )
        .with_columns(
            pl.col("start_run").cum_sum().over("household").alias("run_id")
        )
    )

    inactive_ep = (
        inactive_runs
        .filter(pl.col("inactive"))
        .group_by(["household", "run_id"])
        .agg([
            pl.col("ts").min().alias("start"),
            pl.col("ts").max().alias("end"),
            pl.count().alias("n_slots"),
        ])
        .with_columns(
            (pl.col("n_slots") / 48).alias("days")  # 48 half-hours/day
        )
    )

    inactive_ep = inactive_ep.with_columns(
        (pl.col("days") >= PROLONGED_DAYS).alias("prolonged")
    )

    # list_inactive = inactive_ep['household'].unique().to_list()

    # inactive_runs.filter(pl.col('inactive'))

    # print(inactive_ep.filter(pl.col('prolonged')).sort(pl.col('days'))['household'].unique())

    ## Removing them from main dataset

    inactive_mask = (
        inactive_runs
        .join(
            inactive_ep.filter(pl.col("prolonged"))
            .select(["household", "run_id"]),
            on=["household", "run_id"],
            how="inner"
        )
        .select(["household", "ts"])
    )
    df_clean = df.join(
        inactive_mask,
        on=["household", "ts"],
        how="anti"
    )
    
    return df_clean, inactive_mask

In [ ]:
def find_charge_events(df_post_ev, Q = 0.99): # threshold quantile

    # high quantile method

    cols = ["household", "is_weekend", "season", 'time_period']

    ev_threshold = (
        df_post_ev
        .filter(pl.col('inactive') == False) ## can be removed
        .group_by(cols)
        .agg(pl.col("kwh").quantile(Q, interpolation="nearest").alias("ev_threshold"))  
    )

    df_flagged = (
        df_post_ev
        .join(ev_threshold, on=cols, how="left")
        .with_columns(
            (
                pl.col("kwh")
                > pl.max_horizontal(pl.col("ev_threshold"), pl.lit(3))
            ).alias("ev_like")
        )
    )

    charges = (
        df_flagged
        .filter(pl.col("ev_like"))
        .sort(["household", "ts"])
    )
    
    # detect new episode when gap > 1 hour
    charges = charges.with_columns(
        (
            (pl.col("ts")
            - pl.col("ts").shift(1).over("household")
        ).dt.total_minutes()/60).fill_null(1).alias("gap_h")
    )
    
    charges = charges.with_columns(
        ((pl.col("gap_h") > 1).cast(pl.Int8)).alias("new_episode")
    )
    
    charges = charges.with_columns(
        (pl.col("new_episode").cum_sum().over("household") + 1).alias("episode_id")
    )

    df_join = df_flagged.join(charges, on=cols +["inactive", "ts", "ev_like", "kwh", "ev_threshold"], how="left")

    return df_join

In [ ]:
def remove_low_signal_households(df_flagged, INACTIVE_THRESH=0.90, MAX_EV_EVENTS=20):
    """
    Remove households where:
    - more than INACTIVE_THRESH of rows are inactive
    - and total EV-like events are fewer than MAX_EV_EVENTS
    """

    households_to_remove = (
        df_flagged
        .group_by("household")
        .agg([
            pl.col("inactive").cast(pl.Float64).mean().alias("inactive_share"),
            pl.col("ev_like").cast(pl.Int64).sum().alias("n_ev_events"),
        ])
        .filter(
            (pl.col("inactive_share") > INACTIVE_THRESH) |
            (pl.col("n_ev_events") < MAX_EV_EVENTS)
        )
        .select("household")
    )

    df_clean = df_flagged.join(
        households_to_remove,
        on="household",
        how="anti"
    )

    return df_clean


In [ ]:
# def detect_ev_adoption_weekly(
#     df_flagged,
#     PRE_WINDOW=3,
#     POST_PERSIST=7,
#     JUMP_DIFF=2.5,
#     EARLY_WEEKS=4,          # if any EV in first 3 weeks -> EV from start
#     QUIET_WEEKS=1,          # look only at last 6 weeks before candidate
#     MAX_PRE_ACTIVE=1,       # allow at most 1 active week in quiet window
#     MIN_POST_ACTIVE=1      # require at least 4 active weeks after candidate
# ):
#     weekly = (
#         df_flagged
#         .group_by(["household", pl.col("ts").dt.truncate("1w").alias("week")])
#         .agg(pl.col("ev_like").sum().alias("ev_events"))
#         .sort(["household", "week"])
#     )

#     ev_starts = []

#     for h in weekly["household"].unique():
#         w = weekly.filter(pl.col("household") == h).sort("week")

#         events = w["ev_events"].to_list()
#         weeks = w["week"].to_list()

#         if len(events) == 0:
#             continue

#         found = None

#         # EV from start if there is EV activity in the first few observed weeks
#         if any(e >= 1 for e in events[:EARLY_WEEKS]):
#             continue

#         start_i = max(PRE_WINDOW, QUIET_WEEKS)

#         for i in range(start_i, len(events) - POST_PERSIST + 1):
#             pre = events[i - PRE_WINDOW:i]
#             quiet_pre = events[i - QUIET_WEEKS:i]
#             post = events[i:i + POST_PERSIST]

#             pre_mean = np.mean(pre)
#             post_mean = np.mean(post)

#             pre_active = sum(e >= 1 for e in quiet_pre)
#             post_active = sum(e >= 1 for e in post)

#             if (
#                 post_mean >= pre_mean + JUMP_DIFF
#                 and pre_active <= MAX_PRE_ACTIVE
#                 and post_active >= MIN_POST_ACTIVE
#             ):
#                 found = weeks[i]
#                 break

#         if found is not None:
#             ev_starts.append((h, found))

#     return pl.DataFrame({
#         "household": [h for h, ts in ev_starts],
#         "ev_start_ts": pl.Series([ts for h, ts in ev_starts], dtype=pl.Datetime),
#     }).sort("household")

In [ ]:
# def detect_ev_adoption_weekly(
#     df_flagged,
#     PRE_WINDOW=5,
#     POST_PERSIST=7,
#     JUMP_DIFF=2.5,
# ):
#     # Count EV-like rows per week
#     weekly = (
#         df_flagged
#         .group_by(["household", pl.col("ts").dt.truncate("1w").alias("week")])
#         .agg(pl.col("ev_like").sum().alias("ev_events"))
#         .sort(["household", "week"])
#     )

#     ev_starts = []

#     for h in weekly["household"].unique():
#         w = weekly.filter(pl.col("household") == h).sort("week")

#         events = w["ev_events"].to_list()
#         weeks = w["week"].to_list()

#         if len(events) == 0:
#             print('No charging')
#             continue

#         found = None


#     # If first observed week already has EV events -> EV from start
#         if any(e >= 1 for e in events[:2]):
#             # print('EV from start')
#             continue

#         else:

#             for i in range(PRE_WINDOW, len(events) - POST_PERSIST + 1):
#                 pre = events[i-PRE_WINDOW:i]
#                 post = events[i:i+POST_PERSIST]

#                 pre_mean = np.mean(pre)
#                 post_mean = np.mean(post)

#                 if (
#                     post_mean >= pre_mean + JUMP_DIFF
#                 ):
#                     found = weeks[i]
#                     break

#         if found is not None:
#             ev_starts.append((h, found))

#     return pl.DataFrame({
#         "household": [h for h, ts in ev_starts],
#         "ev_start_ts": pl.Series([ts for h, ts in ev_starts], dtype=pl.Datetime),
#     }).sort("household")


In [ ]:
def detect_ev_adoption_weekly(df_flagged):
    return (
        df_flagged
        .filter(pl.col("episode_id").is_not_null())
        .group_by("household")
        .agg(
            pl.col("ts").min().alias("ev_start_ts")
        )
        .sort("household")
    )

### modelling functions

In [ ]:
def gen_df_events(df_flagged):

    charges = (
        df_flagged
        .filter(pl.col("ev_like"))
        .sort(["household", "ts"])
    )

    # detect new episode when gap > 1 hour
    charges = charges.with_columns(
        (
            (pl.col("ts")
            - pl.col("ts").shift(1).over("household")
        ).dt.total_minutes()/60).fill_null(1).alias("gap_h")
    )

    charges = charges.with_columns(
        (pl.col("gap_h") > 1).cast(pl.Int8).alias("new_episode")
    )

    charges = charges.with_columns(
        pl.col("new_episode").cum_sum().over("household").alias("episode_id")
    )

    episodes = (
        charges
        .group_by(["household", "episode_id"])
        .agg([
            pl.col("ts").min().alias("start_ts"),
            (pl.col("ts").max() + pl.duration(minutes=30)).alias("end_ts"),
        ])
        .sort(["household", "start_ts"])
    )

    return episodes

In [ ]:
def gen_duration(charges): # for each household list their charge events by episode id with their start and end
    
    episodes = (
        charges
        .filter(pl.col("episode_id").is_not_null())
        .group_by(["household", "episode_id"])
        .agg([
            pl.col("ts").min().alias("start_ts"),
            (pl.col("ts").max() + pl.duration(minutes=30)).alias("end_ts"),
        ])
        .sort(["household", "start_ts"])
    )

    episodes = episodes.with_columns(
        ((pl.col("end_ts") - pl.col("start_ts")).dt.total_minutes() / 60)
        .alias("duration_hours")
    )
    #.filter(pl.col("duration_hours") > 1)

    return episodes

In [ ]:
def calculate_gaps(episodes): 
    
    gap_times = (
    episodes
    .with_columns(
        pl.col("start_ts").shift(-1).over("household").alias("next_start")
    )
    .filter(pl.col("next_start").is_not_null())
    .with_columns(
        ((pl.col("next_start") - pl.col("end_ts")).dt.total_minutes()/60).alias("gap_hours_till_next_charge")
    )
    # .select(["household", "start_ts", "end_ts", "next_start", "gap_hours_till_next_charge"])
    )

    return gap_times

### visualization functions

In [ ]:
def compute_freq(df): ## for visu
    cols = ["household", "is_weekend", "season", 'time_period']
    # Just the times to highlight
    charge_events = df.filter(pl.col("ev_like")).select(
        cols + ["ts", "kwh", "ev_threshold"]
    )

    plt.hist(charge_events['ev_threshold'], alpha=0.5)
    plt.title('Distribution of values above which we consider a charging event')

    # frequency
    freq_events = charge_events.group_by(["household"]).agg([
                pl.col("ts").count().alias("nb_of_events"),
                (pl.col('ts').max() - pl.col("ts").min()).alias("data_duration"),
                ((pl.col('ts').max() - pl.col("ts").min()).dt.total_days()/pl.col("ts").count()).alias("freq"),
            ]).sort(["freq", "nb_of_events"])

    print(freq_events)

    return freq_events

In [ ]:
def plot_client_with_charging(
    df_flagged: pl.DataFrame,
    household_name: str,
    prolonged_spans: pl.DataFrame | None = None,
    adoption_df: pl.DataFrame | None = None
):
    d = (
        df_flagged
        .filter(pl.col("household") == household_name)
        .select(["ts", "kwh", "ev_like"])
        .sort("ts")
        .to_pandas()
    )

    _, ax = plt.subplots(figsize=(12, 4))
    ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)")

    # highlight flagged hours
    m = d["ev_like"].astype(bool)
    ax.scatter(d.loc[m, "ts"], d.loc[m, "kwh"], color="red", s=12, label="Charging behaviour")

    # --- shade prolonged inactive spans ---
    # if prolonged_spans is not None:
    #     spans = (
    #         prolonged_spans
    #         .filter(pl.col("household") == household_name)
    #         .select(["start", "end", "days"])
    #         .sort("start")
    #         .to_pandas()
    #     )

    #     added_label = False
    #     for _, r in spans.iterrows():
    #         ax.axvspan(
    #             r["start"], r["end"],
    #             color="gray",
    #             alpha=0.22,
    #             lw=0,
    #             label="Prolonged inactive" if not added_label else None
    #         )
    #         added_label = True
    
# add vertical line for EV adoption date
    if adoption_df is not None:
        ad = adoption_df.filter(pl.col("household") == household_name)

        if ad.height > 0:
            ev_start_ts = ad["ev_start_ts"][0]
            ax.axvline(
                ev_start_ts,
                color="green",
                linestyle="--",
                linewidth=5,
                label="Possible EV adoption"
            )


    ax.set_title(f"Household {household_name}: consumption with EV charging highlighted")
    ax.set_ylabel("kWh per hour")
    ax.legend()
    plt.tight_layout()
    plt.show()

### unused functions

In [ ]:
#Clients with big quantile 95 plateaux 
#from the graph lets choose to consider a jump from 2.7 
#wich is reasonable cuz usually CHARGER_KW_OPTIONS = [2.3, 3.6, 7.0] 

def remove_pre_EV(df, EV_JUMP = 2):  # kWh or equivalent monthly peak jump granny charging is common for Early EV adopters
    
    diff_q = (
        df
        .sort("ts")
        .group_by_dynamic(
            index_column="ts",
            every="1w",
            closed="left",
            group_by="household"
        )
        .agg(
            pl.col("kwh").quantile(0.95).alias("q95")
        ) ### find max of all monthly maximums to see if a big jump happens (also min of all max course)
        .sort(["household", "ts"])
    )

    diff_q = diff_q.with_columns(
        pl.col("q95").diff().over("household").alias("dq95")
    )

    diff_q = diff_q.with_columns(
        (pl.col("dq95") > EV_JUMP).alias("jump_candidate")
    )

    diff_q = diff_q.with_columns(
        pl.col("q95")
        .rolling_mean(13) ## duration of a season
        .shift(-12) # to compute forward
        .over("household")
        .alias("q95_smooth")
    )

    diff_q = diff_q.with_columns(
        (
            pl.col("jump_candidate")
            &
            (
                pl.col("q95_smooth")
                >
                pl.col("q95").shift(1).over("household") + EV_JUMP
            )
        ).alias("ev_adopted")
    )

    ev_start = (
        diff_q
        .filter(pl.col("ev_adopted"))
        .group_by("household")
        .agg(pl.col("ts").min().alias("ev_start_ts"))
    )

    # list_converted = ev_start['household'].unique().to_list()

    ## Removing them from main dataset

    pre_ev_mask = (
        df
        .join(ev_start, on="household", how="inner")
        .filter(pl.col("ts") < pl.col("ev_start_ts"))
    )
    
    df_post_ev = (
        df.join(pre_ev_mask, on=["household", "ts"], how="anti")
        .join(ev_start.select(["household", "ev_start_ts"]), on="household", how="left")
    )

    ### FOR CHECK UPS

    # Every household starts at its EV adoption time
    check = (
        df_post_ev
        .group_by("household")
        .agg(pl.col("ts").min().alias("first_ts"))
        .join(ev_start, on="household")
        .with_columns(
            (pl.col("first_ts") == pl.col("ev_start_ts")).alias("ok") #after cleaning, does each household start exactly at ev_start_ts?
        )
    )

    bad = check.filter(~pl.col("ok")).select(
    "household", "first_ts", "ev_start_ts"
    )

    print(bad)

    print(check.filter(~pl.col("ok")).is_empty())

    return df_post_ev

In [ ]:
# def calculate_duration(df_flagged):

#     charges = (
#         df_flagged
#         .filter(pl.col("ev_like"))
#         .sort(["household", "ts"])
#     )

#     # detect new episode when gap > 1 hour
#     charges = charges.with_columns(
#         (
#             (pl.col("ts")
#             - pl.col("ts").shift(1).over("household")
#         ).dt.total_minutes()/60).fill_null(1).alias("gap_h")
#     )

#     charges = charges.with_columns(
#         (pl.col("gap_h") > 1).cast(pl.Int8).alias("new_episode")
#     )

#     charges = charges.with_columns(
#         pl.col("new_episode").cum_sum().over("household").alias("episode_id")
#     )

#     episodes = (
#         charges
#         .group_by(["household", "episode_id"])
#         .agg([
#             pl.col("ts").min().alias("start_ts"),
#             (pl.col("ts").max() + pl.duration(minutes=30)).alias("end_ts"),
#         ])
#         .sort(["household", "start_ts"])
#     )

#     return episodes

# Exploration

### Cleaning and looking

In [ ]:
df_C = clean_df('controle', False)
df_I = clean_df('intervention', False)

df_preC = clean_df('controle')
df_preI = clean_df('intervention')

In [ ]:
df_preC.head()

In [ ]:
df_C.head()

In [ ]:
print(df_C['household'].unique().shape,
      df_I['household'].unique().shape,
      df_preC['household'].unique().shape,
      df_preI['household'].unique().shape)

In [ ]:
# Real data duration

# clients_duration = df_C.filter(pl.col('kwh') > 0.1).group_by(['household']).agg([
#             (pl.col("ts").max() - pl.col("ts").min()).alias("no_zero_duration"),
#         ]).sort('no_zero_duration')

# clients_duration

## Removing unexploitable data 

### Inactive or not?

In [ ]:
df_C_clean = remove_inactive(df_C)[0]
df_I_clean = remove_inactive(df_I)[0]

In [ ]:
df_preC_clean = remove_inactive(df_preC)[0]
df_preI_clean = remove_inactive(df_preI)[0]

In [ ]:
remove_inactive(df_preI)[1]

#### Plots bonanza

In [ ]:
# df = df_I
# for i, household_name in enumerate(remove_inactive(df)[1]['household'].unique()):
#     d = df.filter(pl.col("household") == household_name).select(["ts", "kwh"]).sort("ts").to_pandas()

#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)")

#     ax.set_title(f"Trial example {i+1} of household in Intervention with inactive periods")

In [ ]:
# df = df_C
# for i, household_name in enumerate(remove_inactive(df)[1]['household'].unique()):
#     d = df.filter(pl.col("household") == household_name).select(["ts", "kwh"]).sort("ts").to_pandas()

#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)")

#     ax.set_title(f"Trial example {i+1} of household in Control with inactive periods")

In [ ]:
# df = df_preI
# for i, household_name in enumerate(remove_inactive(df)[1]['household'].unique()):
#     d = df.filter(pl.col("household") == household_name).select(["ts", "kwh"]).sort("ts").to_pandas()

#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)")

#     ax.set_title(f"Pretrial example {i+1} of household in Intervention with inactive periods")

In [ ]:
# df = df_preC
# for i, household_name in enumerate(remove_inactive(df)[1]['household'].unique()):
#     d = df.filter(pl.col("household") == household_name).select(["ts", "kwh"]).sort("ts").to_pandas()

#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)")

#     ax.set_title(f"Pretrial example {i+1} of household in Control with inactive periods")

### EV converted or not?

#### nb of episode jump method

In [ ]:
# # Distribution of episodes per time period through time for each household

# time_period = "1w"

# df_plot = (
#     df_flagged_C
#         .sort("ts")
#         .group_by_dynamic(
#             index_column="ts",
#             every=time_period,
#             closed="left",
#             group_by="household"
#         )
#         .agg(
#             pl.col("episode_id").count().alias("nb_of_episodes")
#         )
#         .sort(["household", "ts"]))

# q = (
#     df_plot
#     .group_by("household")
#     .agg([
#         pl.col("nb_of_episodes").quantile(0.15).alias("q25"),
#         pl.col("nb_of_episodes").quantile(0.85).alias("q75"),
#     ])
#     .with_columns((pl.col("q75") - pl.col("q25")).alias("IQR")))

# # plt.hist(q['IQR'])

# q_filtered = q.filter(pl.col("IQR") >= 2.7)  ## is there a jump in the numbers of ep

# # iqr_threshold = q.select(pl.col("IQR").quantile(0.25)).item()
# # q_filtered = q.filter(pl.col("IQR") >= iqr_threshold)

# df_wide = df_plot.join(q_filtered.select("household"), on="household", how="inner")

# q

In [ ]:
## Which clients have a jump in episodes distribution?
#df_flagged_C.filter(pl.col("household").is_in(df_wide["household"].unique())).to_pandas().groupby("household")

# for household, group in df_flagged_C.group_by("household"):
#     print(household)
#     plt.plot(group["ts"], group["kwh"])
#     plt.show()
    
#     plt.close(fig)

In [ ]:
# for household, group in df_wide.to_pandas().groupby("household"):
#     plt.plot(group["ts"], group["nb_of_episodes"])


#### extra method

In [ ]:
# from scipy import stats
# def find_adoption_changepoint(hh_data, consumption_col='kwh', min_periods=600): #1.7 semaine
#     """Find the point where consumption distribution shifts significantly."""

#     has_ev = {}
#     hh_data = pd.DataFrame(df_flagged_C, columns=df_flagged_C.columns)

#     for house in set(hh_data["household"]): 
        
#         df_house = hh_data[hh_data["household"] == house]
#         values = df_house[consumption_col].values
#         n = len(values)
        
#         # if n < min_periods * 2:
#         #     return None
        
#         best_idx, best_stat = None, 0
        
#         for i in range(min_periods, n - min_periods):
#             before, after = values[:i], values[i:]
#             # Compare variance or upper quantiles
#             stat = np.percentile(after, 95) - np.percentile(before, 95)
#             if stat > best_stat:
#                 best_stat, best_idx = stat, i
        
#         # Return date if change is significant enough
#         if best_stat > 2:
#             has_ev[house] = df_house.iloc[best_idx]['ts']
    
#     return has_ev

In [ ]:
# for i, house in enumerate(set(df_flagged_C['household'])): 
#     res = find_adoption_changepoint(df_flagged_C.filter(pl.col('household')== house), min_periods=100)
    
#     if res != None:
#         print(res)
#         print(i)
#         break

#### final method

##### finding charging events

In [ ]:
df_flagged_C = find_charge_events(df_C_clean)
df_flagged_I = find_charge_events(df_I_clean)

df_flagged_preC = find_charge_events(df_preC_clean)
df_flagged_preI = find_charge_events(df_preI_clean)

##### remove low signal

In [ ]:
# Remove households with too much inactivity + too little EV-like signal
df_flagged_C = remove_low_signal_households(df_flagged_C)
df_flagged_I = remove_low_signal_households(df_flagged_I)

df_flagged_preC = remove_low_signal_households(df_flagged_preC)
df_flagged_preI = remove_low_signal_households(df_flagged_preI)

##### Removing pre EV

In [ ]:
# gaps between high plateaux? JUST FOR VISU

df = (
    df_flagged_C
    .sort("ts")
    .group_by_dynamic(
        index_column="ts",
        every="1mo",
        closed="left",
        group_by="household"
    )
    .agg(
        pl.col("kwh").quantile(0.95).alias("q95")
    ) ### find max of all monthly maximums to see if a big jump happens (also min of all max course)
    .group_by("household")
    .agg([
        (pl.col("q95").max()-pl.col("q95").min()).alias("diff_q95"), 
    ])
)

plt.hist(df['diff_q95'])
plt.title('Distribution of consumption plateaux differences across all households')

In [ ]:
# Detect households with EV adoption and outpu start date NO CONSUMPTION
df_C_post_ev = detect_ev_adoption_weekly(df_flagged_C)
df_I_post_ev = detect_ev_adoption_weekly(df_flagged_I)

df_preC_post_ev = detect_ev_adoption_weekly(df_flagged_preC)
df_preI_post_ev = detect_ev_adoption_weekly(df_flagged_preI)

In [ ]:
def keep_post_ev_rows(df_flagged, df_adoption):
    return (
        df_flagged
        .join(df_adoption, on="household", how="left")
        .filter(
            pl.col("ev_start_ts").is_null() |
            (pl.col("ts") >= pl.col("ev_start_ts"))
        )
        .drop("ev_start_ts")
    )

df_flagged_C_post_ev = keep_post_ev_rows(df_flagged_C, df_C_post_ev)
df_flagged_I_post_ev = keep_post_ev_rows(df_flagged_I, df_I_post_ev)

df_flagged_preC_post_ev = keep_post_ev_rows(df_flagged_preC, df_preC_post_ev)
df_flagged_preI_post_ev = keep_post_ev_rows(df_flagged_preI, df_preI_post_ev)

In [ ]:
print("C:", df_flagged_C.height)
print("I:", df_flagged_I.height)
print("preC:", df_flagged_preC.height)
print("preI:", df_flagged_preI.height)

In [ ]:
print("C:", df_flagged_C_post_ev.height)
print("I:", df_flagged_I_post_ev.height)
print("preC:", df_flagged_preC_post_ev.height)
print("preI:", df_flagged_preI_post_ev.height)

In [ ]:
# gaps between high plateaux? JUST FOR VISU 

# df = (
#     df_C_post_ev
#     .sort("ts")
#     .group_by_dynamic(
#         index_column="ts",
#         every="1mo",
#         closed="left",
#         group_by="household"
#     )
#     .agg(
#         pl.col("kwh").max().alias("window_max")
#     ) ### find max of all monthly maximums to see if a big jump happens (also min of all max course)
#     .group_by("household")
#     .agg([
#         (pl.col("window_max").max()-pl.col("window_max").min()).alias("diff_window_max"), 
#     ])
# )

# plt.hist(df['diff_window_max'])
# plt.title('Distribution of consumption max differences across all households')

# Creation of events df

In [ ]:
episodes_C = gen_duration(df_flagged_C_post_ev)
episodes_I = gen_duration(df_flagged_I_post_ev)

episodes_preC = gen_duration(df_flagged_preC_post_ev)
episodes_preI = gen_duration(df_flagged_preI_post_ev)

In [ ]:
# episodes_C = gen_df_events(df_flagged_C)
# episodes_I = gen_df_events(df_flagged_I)

# episodes_preC = gen_df_events(df_flagged_preC)
# episodes_preI = gen_df_events(df_flagged_preI)

## visu

### duree

In [ ]:
gen_duration(df_flagged_C)

In [ ]:
gen_duration(df_flagged_C).group_by(["household"]).agg(pl.col("duration_hours").count().alias("total_charging"))


### freq de charge

In [ ]:
charge_events_C = compute_freq(df_flagged_C)
charge_events_I = compute_freq(df_flagged_I)

In [ ]:
charge_events_preC = compute_freq(df_flagged_preC)
charge_events_preI = compute_freq(df_flagged_preI)

### Plotting charge events

In [ ]:
# Trial Control
for household in df_C_post_ev['household'].unique():
    plot_client_with_charging(df_flagged_C, household, adoption_df = df_C_post_ev)

In [ ]:
# this is cool but ugly 
# --- prerequisites ---
# df_flagged_C: Polars DataFrame with columns ["ts","household","kwh", ...]
# detect_ev_adoption_weekly: function we fixed earlier returning Polars DF
# with schema {"household": pl.Utf8, "ev_start_ts": pl.Datetime}


# df = df_flagged_C 

# # 1) Compute EV start dates ONCE
# ev_starts = detect_ev_adoption_weekly(df)  # Polars DF: household, ev_start_ts

# # 2) Iterate households that have an EV start detected
# households = df_flagged_C.get_column("household").to_list()

# for i, household_name in enumerate(households, start=1):

#     # ---- raw series for this household (to pandas) ----
#     d = (
#         df
#         .filter(pl.col("household") == household_name)
#         .select(["ts", "kwh"])
#         .sort("ts")
#         .to_pandas()
#     )

#     # Safety: skip empty / all-null
#     if d.empty:
#         continue

#     # ---- 2-week quantile (q25) for visual context, computed in Polars then to pandas ----
#     q25_tw = (
#         df
#         .filter(pl.col("household") == household_name)
#         .group_by_dynamic(index_column="ts", every="3w", closed="left")
#         .agg(pl.col("kwh").quantile(0.4).alias("q25"))
#         .sort("ts")
#         .to_pandas()
#     )

#     # ---- get EV start timestamp for this household as pandas.Timestamp ----
#     ev_ts_polars = (
#         ev_starts
#         .filter(pl.col("household") == household_name)
#         .select("ev_start_ts")
#         .to_series()
#     )
#     if ev_ts_polars.is_empty():
#         # if no start (shouldn't happen because we iterated 'households')
#         continue
#     ev_ts = pd.to_datetime(ev_ts_polars.to_list()[0])  # pandas.Timestamp

#     # ---- PLOTTING ----
#     fig, ax = plt.subplots(figsize=(12, 4))

#     # raw consumption
#     ax.plot(d["ts"], d["kwh"], label="Consumption (kWh)", color="#1f77b4", alpha=0.6, lw=1)

#     # 2-week q25 line
#     if not q25_tw.empty:
#         ax.plot(q25_tw["ts"], q25_tw["q25"], label="2-week q25", color="#2ca02c", lw=2)

#     # vertical line at EV start
#     ax.axvline(ev_ts, color="red", linestyle="--", linewidth=2, label="EV adoption start")

#     # (optional) shade pre-EV period
#     ax.axvspan(d["ts"].min(), ev_ts, color="grey", alpha=0.12, label="Pre-EV window")

#     ax.set_title(f"Example {i} — Intervention household with pre-EV period")
#     ax.set_ylabel("kWh")
#     ax.legend(loc="upper right")
#     ax.grid(alpha=0.2)
#     plt.tight_layout()
#     plt.show()

In [ ]:
# # Trial Control
# for household in df_flagged_C['household'].unique():
#     plot_client_with_charging(df_flagged_C, household, prolonged_spans = remove_inactive(df_C)[0].filter(pl.col('household') == household))

In [ ]:
#PreTrial Control
for household in df_flagged_preC['household'].unique():
    plot_client_with_charging(df_flagged_preC, household)

In [ ]:
#PreTrial Intervention
for household in df_flagged_preI['household'].unique():
    plot_client_with_charging(df_flagged_preI, household)

# Survival models

In [ ]:
episodes_C   # removing events of less than an hour romoes roughly 2300 events on 3200 #um old

In [ ]:
# gap times

##pretrial
gap_times_preC = calculate_gaps(episodes_preC).to_pandas()
gap_times_preC['intervention'] = 0

gap_times_preI = calculate_gaps(episodes_preI).to_pandas()
gap_times_preI['intervention'] = 1

gap_times_pre = pd.concat([gap_times_preC, gap_times_preI])
gap_times_pre['pretrial'] = 1

##trial
gap_times_C = calculate_gaps(episodes_C).to_pandas()
gap_times_C['intervention'] = 0

gap_times_I = calculate_gaps(episodes_I).to_pandas()
gap_times_I['intervention'] = 1

gap_times = pd.concat([gap_times_C, gap_times_I])
gap_times['pretrial'] = 0

# #all
# gap_times = pd.concat([gap_times, gap_times_pre])
# gap_times


In [ ]:
gap_times

In [ ]:
gap_times

## Frequence

### KM plots

In [ ]:
# How long after a charge does the next charge happen?

kmf = ll.KaplanMeierFitter()
kmf.fit(gap_times_C["gap_hours_till_next_charge"], label="Time between EV charging episodes")
kmf.plot_survival_function()

plt.xlabel("Hours since last charge")
plt.ylabel("P(no new charge yet)")
plt.show()

#Steep drop = frequent charging

###  Gap-time model : Andersen–Gill Cox model

### new

In [ ]:
#duree frequence

### PRETRIAL
# Time between end of one charge and start of the next

cph = ll.CoxPHFitter()

cph.fit(
    gap_times_pre[[
    "gap_hours_till_next_charge",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="gap_hours_till_next_charge",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

In [ ]:
### TRIAL
# Time between end of one charge and start of the next

cph = ll.CoxPHFitter()

cph.fit(
    gap_times[[
    "gap_hours_till_next_charge",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="gap_hours_till_next_charge",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

### old

In [ ]:
#duree frequence

### PRETRIAL
# Time between end of one charge and start of the next

cph = ll.CoxPHFitter()

cph.fit(
    gap_times_pre[[
    "gap_hours_till_next_charge",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="gap_hours_till_next_charge",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, 
# compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

In [ ]:
### TRIAL
# Time between end of one charge and start of the next

cph = ll.CoxPHFitter()

cph.fit(
    gap_times[[
    "gap_hours_till_next_charge",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="gap_hours_till_next_charge",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

## Duration

### new

In [ ]:
### PRETRIAL

cph = ll.CoxPHFitter()

cph.fit(
    gap_times_pre[[
    "duration_hours",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="duration_hours",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

In [ ]:
#TRIAL

cph = ll.CoxPHFitter()

cph.fit(
    gap_times[[
    "duration_hours",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="duration_hours",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

### old

In [ ]:
### PRETRIAL

cph = ll.CoxPHFitter()

cph.fit(
    gap_times_pre[[
    "duration_hours",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="duration_hours",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?

In [ ]:
#TRIAL

cph = ll.CoxPHFitter()

cph.fit(
    gap_times[[
    "duration_hours",
    "household",          # cluster
    "intervention",       # example covariate
]],
    duration_col="duration_hours",
    cluster_col="household"
)

cph.print_summary()

# How does the intervention change the rate at which the next charging event occurs, conditional on the previous one?

# After the intervention, households are 12% more likely to start a new EV charging session at any given hour, compared to control --> Charging gaps are shorter under intervention.

#given two charging gaps does the model correctly assign a higher hazard to the one that actually ends sooner?